In [78]:
import numpy as np
import pandas as pd
import os
import re
from sklearn.preprocessing import MinMaxScaler
from scipy import sparse as sp

In [79]:
df = pd.read_csv('dataset11-30.csv')
arh_11_20 = pd.read_csv('export_arh_11-20-final.csv')
arh_21_30 = pd.read_csv('export_arh_21-30-final.csv')
arh_31_42 = pd.read_csv('export_arh_31-42-final.csv')
submission = pd.read_csv('submission.csv')

C:\Users\Maryam\AppData\Local\Temp\ipykernel_26484\424979671.py:4: DtypeWarning: Columns (0: year_of_production) have mixed types. Specify dtype option on import or set low_memory=False.
  arh_31_42 = pd.read_csv('export_arh_31-42-final.csv')


## Data preproccesing

In [80]:
#Turning dates to date format
dt_format = "%Y-%m-%d %H:%M:%S"
df["start_time"] = pd.to_datetime(df['start_time'], format=dt_format)
df["stop_time"] = pd.to_datetime(df['stop_time'], format=dt_format)

dt_format = "%d.%m.%Y %H:%M:%S"
arh_11_20["start_time"] = pd.to_datetime(arh_11_20['start_time'], format=dt_format)
arh_21_30["start_time"] = pd.to_datetime(arh_21_30['start_time'], format=dt_format)
arh_31_42["start_time"] = pd.to_datetime(arh_31_42['start_time'], format=dt_format)

In [81]:
#Checking time data
print("11-20:", arh_11_20["start_time"].min(), arh_11_20["start_time"].max())
print("Days:", (arh_11_20["start_time"].max() - arh_11_20["start_time"].min()).days)
print("21-30:", arh_21_30["start_time"].min(), arh_21_30["start_time"].max())
print("Days:", (arh_21_30["start_time"].max() - arh_21_30["start_time"].min()).days)
print("31-42:", arh_31_42["start_time"].min(), arh_31_42["start_time"].max())
print("Days:", (arh_31_42["start_time"].max() - arh_31_42["start_time"].min()).days)

11-20: 2020-03-08 23:00:00 2020-05-18 11:45:00
Days: 70
21-30: 2020-05-18 00:00:00 2020-07-27 11:55:00
Days: 70
31-42: 2020-07-27 00:00:00 2020-10-19 11:55:00
Days: 84


In [82]:
def prepare_schedule(schedule_raw):
    """
    Prepare channels' schedule to search show by time
    """
    schedule = schedule_raw[["channel_id", "start_time", "duration", "tv_show_id"]].copy()
    schedule = schedule[schedule["tv_show_id"] != 0]
    schedule = schedule.rename(columns={"start_time": "show_start", "duration": "show_duration"})
    schedule["show_stop"] = schedule["show_start"] + pd.to_timedelta(schedule["show_duration"], unit="s")
    return schedule.sort_values(["channel_id", "show_start"]).reset_index(drop=True)

In [83]:
def prepare_sessions(schedule_raw):
    """
    Prepare viewing sessions: user_id, channel_id, view_start, view_stop.
    """
    schedule = schedule_raw[["user_id", "vsetv_id", "start_time", "duraton"]].copy()
    schedule = schedule.rename(columns={"start_time": "view_start", "duraton": "view_duration", "vsetv_id": "channel_id"})
    schedule["view_stop"] = schedule["view_start"] + pd.to_timedelta(schedule["view_duration"], unit="s")
    return schedule.sort_values(["channel_id", "view_start"]).reset_index(drop=True)

In [84]:
def flatten_candidate_ranges(first_candidate, one_past_last):
    """
    Converts the index ranges [first_candidate[i], one_past_last[i]) into a flat list of pairs.
    """
    range_lengths = np.maximum(one_past_last - first_candidate, 0)
    total_pairs   = range_lengths.sum()

    pair_owner = np.repeat(np.arange(len(first_candidate)), range_lengths)
    flat_positions = np.arange(total_pairs)
    block_starts = np.cumsum(range_lengths) - range_lengths
    offset = flat_positions - block_starts[pair_owner]

    return pair_owner, offset

In [85]:
def build_facts(sessions, schedule):
    """
    Interval overlap join between sessions and schedule, per channel.

    Returns one row per (session, airing) that overlap in time.
    """
    schedule_by_channel = {ch: g.sort_values('show_start').reset_index(drop=True)for ch, g in schedule.groupby('channel_id')}
    channel_blocks = []

    for channel_id, channel_sessions in sessions.groupby('channel_id', sort=False):
        
        channel_schedule = schedule_by_channel.get(channel_id)
        if channel_schedule is None or channel_schedule.empty:
            continue

        airing_starts     = channel_schedule['show_start'].values
        airing_stops      = channel_schedule['show_stop'].values
        airing_durations  = channel_schedule['show_duration'].values
        airing_show_ids   = channel_schedule['tv_show_id'].values

        session_users  = channel_sessions['user_id'].values
        session_starts = channel_sessions['view_start'].values
        session_stops  = channel_sessions['view_stop'].values

        first_candidate = np.searchsorted(airing_starts, session_starts, side='left')
        one_past_last   = np.searchsorted(airing_starts, session_stops,  side='left')

        has_previous_airing  = first_candidate > 0
        previous_airing_idx  = np.where(has_previous_airing, first_candidate - 1, 0)
        previous_still_airing = has_previous_airing & (airing_stops[previous_airing_idx] > session_starts)
        first_candidate = np.where(previous_still_airing, first_candidate - 1, first_candidate)

        session_of_pair, offset_in_range = flatten_candidate_ranges(first_candidate, one_past_last)
        if session_of_pair.size == 0:
            continue
        airing_of_pair = first_candidate[session_of_pair] + offset_in_range

        view_start  = session_starts[session_of_pair]
        view_stop   = session_stops[session_of_pair]
        show_start  = airing_starts[airing_of_pair]
        show_stop   = airing_stops[airing_of_pair]

        overlap_seconds = (np.minimum(view_stop, show_stop) - np.maximum(view_start, show_start)) / np.timedelta64(1, 's')
        overlaps = overlap_seconds > 0
        if not overlaps.any():
            continue

        session_of_pair = session_of_pair[overlaps]
        airing_of_pair  = airing_of_pair[overlaps]

        channel_blocks.append(pd.DataFrame({
            'user_id':       session_users[session_of_pair],
            'channel_id':    channel_id,
            'view_start':    session_starts[session_of_pair],
            'view_stop':     session_stops[session_of_pair],
            'show_start':    airing_starts[airing_of_pair],
            'show_stop':     airing_stops[airing_of_pair],
            'show_duration': airing_durations[airing_of_pair],
            'tv_show_id':    airing_show_ids[airing_of_pair],
            'overlap_sec':   overlap_seconds[overlaps].astype('int64'),
        }))

    if not channel_blocks:
        return pd.DataFrame(columns=[
            'user_id', 'channel_id', 'view_start', 'view_stop',
            'show_start', 'show_stop', 'show_duration', 'tv_show_id', 'overlap_sec',
        ])
    return pd.concat(channel_blocks, ignore_index=True)

In [86]:
def build_airings(facts):
    """
    Collapse facts to one row per (user, airing).
    view_pct = total overlap / airing duration.
    """
    airings = (facts.groupby(['user_id', 'channel_id', 'show_start', 'show_stop', 'show_duration', 'tv_show_id'], as_index=False)['overlap_sec'].sum().rename(columns={'overlap_sec': 'full_overlap'}))
    airings['view_pct'] = (airings['full_overlap'] / airings['show_duration']).clip(0, 1)
    return airings

In [87]:
def build_user_show(airings, threshold=0.8):
    """
    From airing-level to (user, show) scores
    """
    a = airings

    freq = (a.loc[a['view_pct'] >= threshold].groupby(['user_id', 'tv_show_id'], as_index=False).size().rename(columns={'size': 'frequency'}))

    alt = (a.groupby(['user_id', 'tv_show_id'], as_index=False).agg(sum_view_pct=('view_pct', 'sum'),total_overlap_sec=('full_overlap', 'sum'),total_show_duration=('show_duration', 'sum'),n_airings_watched=('view_pct', 'size'),))

    return alt.merge(freq, on=['user_id', 'tv_show_id'], how='left').fillna({'frequency': 0})

In [88]:
def build_profile(user_show):
    """
    dict[user_id] -> dict[tv_show_id] -> frequency
    Only shows with frequency > 0 are kept
    """
    good = user_show[user_show['frequency'] > 0]
    profile = {}
    for uid, sid, freq in zip(good['user_id'], good['tv_show_id'], good['frequency']):
        profile.setdefault(uid, {})[sid] = int(freq)
    return profile

In [89]:
val_start = pd.Timestamp('2020-05-18')

sessions = prepare_sessions(df)
sessions_train = sessions[sessions['view_start'] <  val_start].reset_index(drop=True)
sessions_val = sessions[sessions['view_start'] >= val_start].reset_index(drop=True)

schedule_train = prepare_schedule(arh_11_20)
schedule_val   = prepare_schedule(arh_21_30)

In [90]:
facts_train = build_facts(sessions_train, schedule_train)
facts_val = build_facts(sessions_val, schedule_val)

In [91]:
airings_train = build_airings(facts_train)
airings_val = build_airings(facts_val)

In [92]:
user_show_train = build_user_show(airings_train)
user_show_val   = build_user_show(airings_val)

In [93]:
profile_train = build_profile(user_show_train)

## Functions for evaluation

In [94]:
def build_top5(user_show, k=5):
    """dict[user_id] -> list[show_id], only users with exactly k shows, frequency>0."""
    good = user_show[user_show['frequency'] > 0].copy()
    good = good.sort_values(['user_id', 'frequency'], ascending=[True, False])
    good = good.groupby('user_id').head(k)
    gt = good.groupby('user_id')['tv_show_id'].apply(list)
    gt = gt[gt.apply(len) == k]
    return gt.to_dict()

In [95]:
def map_at_5(pred, gt):
    """pred, gt: dict[user_id] -> list[show_id]. Mean AP@5 over intersection of users."""
    users = set(pred) & set(gt)
    aps = []
    for u in users:
        hits = 0
        ap = 0.0
        for k, sid in enumerate(pred[u][:5], start=1):
            if sid in gt[u]:
                hits += 1
                ap += hits / k
        aps.append(ap / 5.0)
    return np.mean(aps)

## Popularity (baseline)

In [96]:
def build_popularity(user_show, available_shows=None, top_n=5):
    """
    Сalculates the top‑N most popular shows
    """
    good = user_show[user_show['frequency'] > 0]
    
    if available_shows is not None:
        good = good[good['tv_show_id'].isin(set(available_shows))]

    by_users = good.groupby('tv_show_id')['user_id'].nunique()
    by_freq = good.groupby('tv_show_id')['frequency'].sum()

    scores = pd.DataFrame({'users': by_users, 'freq': by_freq})
    scores = scores.sort_values(['users', 'freq'], ascending=False)
    
    return scores.head(top_n).index.tolist()

In [97]:
val_candidates = arh_21_30.loc[arh_21_30['tv_show_id'] != 0, 'tv_show_id'].unique()
gt_val = build_top5(user_show_val)

popularity_train = build_popularity(user_show_train, available_shows=val_candidates, top_n=5)
print("Popularity baseline top-5:", popularity_train)

pred_val_popularity = {uid: popularity_train for uid in gt_val.keys()}
map_popularity = map_at_5(pred_val_popularity, gt_val)
print("Popularity baseline MAP@5:", map_popularity)

Popularity baseline top-5: [2400480, 240081, 2400467, 200344, 2400508]
Popularity baseline MAP@5: 0.2574160463192722


## Content-Based Filtering

In [98]:
_YEAR_RE = re.compile(r'^(\d{4})(?:-(\d{4}|\?{3}))?(-)?$')

def parse_year(y, data_max_year=2020):
    """Parse year_of_production → (year_start, year_end, year_span, is_ongoing)."""
    if pd.isna(y):
        return (np.nan, np.nan, np.nan, 0)
    if isinstance(y, (int, float, np.integer, np.floating)):
        yr = int(y)
        return (yr, yr, 0, 0) if 1900 <= yr <= data_max_year else (np.nan, np.nan, np.nan, 0)
    s = str(y).strip().replace('–', '-').replace('—', '-').replace(' ', '')
    m = _YEAR_RE.match(s)
    if not m:
        return (np.nan, np.nan, np.nan, 0)
    start, end_raw, ongoing = int(m.group(1)), m.group(2), m.group(3) == '-'
    if end_raw is None:
        end = data_max_year if ongoing else start
        return (start, end, end - start, int(ongoing))
    if end_raw == '???':
        return (start, np.nan, np.nan, 0)
    end = int(end_raw)
    return (start, end, end - start, int(ongoing))

In [99]:
def build_show_features(sched_raw, data_max_year=2020):
    """
    Builds one vector per show

    Returns DataFrame indexed by tv_show_id with columns:
      - tod_morning/day/evening/night
      - is_weekend
      - log_mean_sec, log_min_sec, log_max_sec, log_spread_sec
      - year_start, year_end, year_span, is_ongoing
      - cat_*    (one-hot, 9 categories)
      - genre_*  (one-hot, all genres)
    """
    s = sched_raw.copy()
    s['show_start']    = s['start_time']
    s['show_duration'] = s['duration']

    hour = s['show_start'].dt.hour
    s['tod_morning'] = ((hour >= 6)  & (hour < 12)).astype(int)
    s['tod_day']     = ((hour >= 12) & (hour < 18)).astype(int)
    s['tod_evening'] = ((hour >= 18) & (hour < 24)).astype(int)
    s['tod_night']   = (hour < 6).astype(int)
    s['is_weekend']  = (s['show_start'].dt.dayofweek >= 5).astype(int)

    agg = s.groupby('tv_show_id').agg(
        mean_sec    = ('show_duration', 'mean'),
        min_sec     = ('show_duration', 'min'),
        max_sec     = ('show_duration', 'max'),
        tod_morning = ('tod_morning', 'mean'),
        tod_day     = ('tod_day',     'mean'),
        tod_evening = ('tod_evening', 'mean'),
        tod_night   = ('tod_night',   'mean'),
        is_weekend  = ('is_weekend',  'mean'),
    ).reset_index()
    agg['spread_sec'] = agg['max_sec'] - agg['min_sec']
    for c in ['mean_sec', 'min_sec', 'max_sec', 'spread_sec']:
        agg['log_' + c] = np.log1p(agg[c])
    agg = agg.drop(columns=['mean_sec', 'min_sec', 'max_sec', 'spread_sec'])

    meta = s.drop_duplicates('tv_show_id')[
        ['tv_show_id', 'tv_show_category',
         'tv_show_genre_1', 'tv_show_genre_2', 'tv_show_genre_3',
         'year_of_production']
    ].reset_index(drop=True)

    year_rows = meta['year_of_production'].apply(parse_year)
    year_df = pd.DataFrame(
        year_rows.tolist(), index=meta['tv_show_id'].values,
        columns=['year_start', 'year_end', 'year_span', 'is_ongoing']
    ).reset_index().rename(columns={'index': 'tv_show_id'})
    for c in ['year_start', 'year_end', 'year_span']:
        year_df[c] = year_df[c].fillna(year_df[c].median())

    cat = pd.get_dummies(meta['tv_show_category'], prefix='cat').astype(int)
    cat.insert(0, 'tv_show_id', meta['tv_show_id'].values)

    long = pd.concat([
        meta[['tv_show_id', 'tv_show_genre_1']].rename(columns={'tv_show_genre_1': 'g'}),
        meta[['tv_show_id', 'tv_show_genre_2']].rename(columns={'tv_show_genre_2': 'g'}),
        meta[['tv_show_id', 'tv_show_genre_3']].rename(columns={'tv_show_genre_3': 'g'}),
    ], ignore_index=True).dropna()
    long = long.drop_duplicates(['tv_show_id', 'g'])
    genre = (
        long.assign(v=1)
            .pivot(index='tv_show_id', columns='g', values='v')
            .fillna(0).astype(int)
            .add_prefix('genre_')
            .reset_index()
    )

    result = (agg.merge(year_df, on='tv_show_id', how='left')
                 .merge(cat,     on='tv_show_id', how='left')
                 .merge(genre,   on='tv_show_id', how='left'))
    onehot = [c for c in result.columns if c.startswith(('cat_', 'genre_'))]
    result[onehot] = result[onehot].fillna(0).astype(int)
    return result.set_index('tv_show_id')

In [100]:
all_sched = pd.concat([arh_11_20, arh_21_30, arh_31_42], ignore_index=True)
all_sched = all_sched[all_sched['tv_show_id'] != 0].reset_index(drop=True)
show_features = build_show_features(all_sched)

continuous_cols = [c for c in show_features.columns if not c.startswith(('cat_', 'genre_'))]
onehot_cols = [c for c in show_features.columns if c.startswith(('cat_', 'genre_'))]

show_vec = show_features.copy()
show_vec[continuous_cols] = MinMaxScaler().fit_transform(show_features[continuous_cols])

In [101]:
def build_user_vectors(profile, show_vec):
    """
    profile: dict[user_id][show_id] -> frequency
    Returns DataFrame (users × features)
    """
    rows, users = [], []
    for uid, shows in profile.items():
        idx = [s for s in shows if s in show_vec.index]
        if not idx:
            continue
        w = np.array([shows[s] for s in idx], dtype=float)
        V = show_vec.loc[idx].values
        rows.append((V * w[:, None]).sum(0) / w.sum())
        users.append(uid)
    return pd.DataFrame(rows, index=users, columns=show_vec.columns)

In [102]:
def top_k_candidates(user_vec, candidate_ids, show_vec, k=5):
    """
    Returns dict[user_id] -> list[show_id] (top-k by cosine)
    """
    cand = [s for s in candidate_ids if s in show_vec.index]
    C = show_vec.loc[cand].values
    C_norm = np.linalg.norm(C, axis=1) + 1e-12
    out = {}
    for uid, u in user_vec.iterrows():
        u = u.values
        u_norm = np.linalg.norm(u) + 1e-12
        scores = (C @ u) / (C_norm * u_norm)
        top = np.argsort(-scores, kind='stable')[:k]
        out[uid] = [cand[i] for i in top]
    return out

In [133]:
user_vec_train = build_user_vectors(profile_train, show_vec)
val_candidates = arh_21_30.loc[arh_21_30['tv_show_id'] != 0, 'tv_show_id'].unique()
pred_val_cbf = top_k_candidates(user_vec_train, val_candidates, show_vec, k=5)

gt_val = build_top5(user_show_val)
map_cbf = map_at_5(pred_val_cbf, gt_val)
print("CBF MAP@5:", map_cbf)

CBF MAP@5: 0.13290690841937378


In [134]:
user_vec_train = build_user_vectors(profile_train, show_vec)

train_watched_shows = set(user_show_train[user_show_train['frequency'] > 0]['tv_show_id'].unique())

val_schedule_shows = set(arh_21_30.loc[arh_21_30['tv_show_id'] != 0, 'tv_show_id'].unique())

val_candidates = list(train_watched_shows.intersection(val_schedule_shows))


pred_val_cbf = top_k_candidates(user_vec_train, val_candidates, show_vec, k=5)
pred_val_cbf = fill_cold_start(pred_val_cbf, gt_val.keys(), popularity_train)
map_cbf_fair = map_at_5(pred_val_cbf, gt_val)
print("CBF MAP@5 (fair pool):", map_cbf_fair)

CBF MAP@5 (fair pool): 0.14735318444995862


## Association Rules (Support)

In [106]:
def fit_support_rules(R):
    """
    S[i, j] = how many users watched both show i and show j
    """
    R_binary = R.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)
    return (R_binary.T @ R_binary).tocsr()

In [107]:
def recommend_top_k(R, user_ids, item_ids, S, candidates, k=5):
    """
    For each user — top-k show candidates based on scoring
    """
    candidates_set = set(candidates)
    cand_positions = [i for i, s in enumerate(item_ids) if s in candidates_set]
    cand_shows = [item_ids[i] for i in cand_positions]

    scores = (R @ S).tocsc()[:, cand_positions].toarray()

    out = {}
    for row, uid in enumerate(user_ids):
        row_scores = scores[row]
        if len(row_scores) <= k:
            top = np.argsort(-row_scores)
        else:
            top = np.argpartition(-row_scores, k - 1)[:k]
            top = top[np.argsort(-row_scores[top])]
        out[uid] = [cand_shows[i] for i in top[:k]]
    return out

In [108]:
def build_baskets(profile, min_frequency=1, weight='frequency'):
    """
    profile: dict[user_id][show_id] -> frequency (from build_profile)
    Returns: dict[user_id][show_id] -> weight, only shows with frequency >= min_frequency
    """
    baskets = {}
    for user, shows in profile.items():
        filtered = {s: f for s, f in shows.items() if f >= min_frequency}
        if not filtered:
            continue
        if weight == 'frequency':
            baskets[user] = filtered
        elif weight == 'binary':
            baskets[user] = {s: 1 for s in filtered}
        elif weight == 'log':
            baskets[user] = {s: float(np.log1p(f)) for s, f in filtered.items()}
    return baskets

In [109]:
from scipy.sparse import csr_matrix

def build_user_item_matrix(baskets):
    """
    baskets: dict[user_id][show_id] -> weight
    Returns: (R_sparse, user_ids, item_ids)
      R_sparse: csr_matrix (n_users, n_items), R[u, i] = weight
      user_ids: sorted list of users
      item_ids: sorted list of shows
    """
    user_ids = sorted(baskets.keys())
    item_ids = sorted({s for shows in baskets.values() for s in shows})
    user_to_idx = {u: i for i, u in enumerate(user_ids)}
    item_to_idx = {s: i for i, s in enumerate(item_ids)}

    rows, cols, vals = [], [], []
    for user, shows in baskets.items():
        for show, weight in shows.items():
            rows.append(user_to_idx[user])
            cols.append(item_to_idx[show])
            vals.append(weight)

    R = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids)))
    return R, user_ids, item_ids

In [110]:
def fill_cold_start(pred, all_users, fallback, k=5):
    out = dict(pred)
    missing = [u for u in all_users if u not in out]
    for uid in missing:
        out[uid] = list(fallback[:k])
    return out

In [111]:
baskets = build_baskets(profile_train, min_frequency=1, weight='frequency')
R, user_ids, item_ids = build_user_item_matrix(baskets)

S_support = fit_support_rules(R)

pred_support = recommend_top_k(R, user_ids, item_ids, S_support, val_candidates, k=5)
pred_support = fill_cold_start(pred_support, gt_val.keys(), popularity_train)

map_ar_support = map_at_5(pred_support, gt_val)
print("AR Support MAP@5:", map_ar_support)

AR Support MAP@5: 0.2849925558312655


## Association Rules (Confidence)

In [112]:
def fit_confidence_rules(R, min_support=5):
    """
    S[i, j] = P(watched j | watched i) = cooc(i, j) / count(i).
    """
    R_binary = R.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)

    cooc = (R_binary.T @ R_binary).tocsr()
    counts = np.asarray(R_binary.sum(axis=0)).flatten()

    inv_counts = np.zeros_like(counts, dtype=np.float32)
    mask = counts >= min_support
    inv_counts[mask] = 1.0 / counts[mask]

    S = sp.diags(inv_counts) @ cooc
    return S.tocsr()

In [113]:
for ms in [1, 3, 5, 10, 20, 50]:
    S_conf = fit_confidence_rules(R, min_support=ms)
    pred_conf = recommend_top_k(R, user_ids, item_ids, S_conf, val_candidates, k=5)
    pred_conf = fill_cold_start(pred_conf, gt_val.keys(), popularity_train)
    map_ar_confidence = map_at_5(pred_conf, gt_val)
    print("min support:", ms, " AR Confidence MAP@5:", map_ar_confidence)

min support: 1  AR Confidence MAP@5: 0.28842597187758473
min support: 3  AR Confidence MAP@5: 0.2882307692307693
min support: 5  AR Confidence MAP@5: 0.28799172870140616
min support: 10  AR Confidence MAP@5: 0.2881033912324235
min support: 20  AR Confidence MAP@5: 0.28836559139784945
min support: 50  AR Confidence MAP@5: 0.2897105045492142


## Association Rules (Lift)

In [114]:
def fit_lift_rules(R, min_support=5):
    """
    S[i, j] = lift(i, j) = cooc(i, j) * N / (count(i) * count(j)).
    """
    R_binary = R.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)

    cooc = (R_binary.T @ R_binary).tocsr()
    counts = np.asarray(R_binary.sum(axis=0)).flatten()
    n_users = R.shape[0]

    inv_counts = np.zeros_like(counts, dtype=np.float32)
    mask = counts >= min_support
    inv_counts[mask] = 1.0 / counts[mask]

    D = sp.diags(inv_counts)
    S = n_users * D @ cooc @ D
    return S.tocsr()

In [115]:
for ms in [5, 10, 20, 50, 100]:
    S_lift = fit_lift_rules(R, min_support=ms)
    pred_lift = recommend_top_k(R, user_ids, item_ids, S_lift, val_candidates, k=5)
    pred_lift = fill_cold_start(pred_lift, gt_val.keys(), popularity_train)
    map_ar_lift = map_at_5(pred_lift, gt_val)
    print("min support:", ms, " AR Lift MAP@5:", map_ar_lift)

min support: 5  AR Lift MAP@5: 0.016322580645161292
min support: 10  AR Lift MAP@5: 0.021450785773366416
min support: 20  AR Lift MAP@5: 0.030722084367245658
min support: 50  AR Lift MAP@5: 0.05040363937138131
min support: 100  AR Lift MAP@5: 0.07034491315136476


In [129]:
ms = 5
S_lift = fit_lift_rules(R, min_support=ms)
pred_lift = recommend_top_k(R, user_ids, item_ids, S_lift, val_candidates, k=5)
pred_lift = fill_cold_start(pred_lift, gt_val.keys(), popularity_train)
map_ar_lift = map_at_5(pred_lift, gt_val)
print("min support:", ms, " AR Lift MAP@5:", map_ar_lift)

min support: 5  AR Lift MAP@5: 0.016322580645161292


## Association Rules (Cosine)

In [116]:
def fit_cosine_rules(R, min_support=5):
    """
    S[i, j] = cooc(i, j) / sqrt(count(i) * count(j)).
    """
    R_binary = R.copy()
    R_binary.data = np.ones_like(R_binary.data, dtype=np.float32)

    cooc = (R_binary.T @ R_binary).tocsr()
    counts = np.asarray(R_binary.sum(axis=0)).flatten()

    inv_sqrt = np.zeros_like(counts, dtype=np.float32)
    mask = counts >= min_support
    inv_sqrt[mask] = 1.0 / np.sqrt(counts[mask])

    D = sp.diags(inv_sqrt)
    S = D @ cooc @ D
    return S.tocsr()

In [117]:
S_cos = fit_cosine_rules(R, min_support=5)
pred_cos = recommend_top_k(R, user_ids, item_ids, S_cos, val_candidates, k=5)
pred_cos = fill_cold_start(pred_cos, gt_val.keys(), popularity_train)
map_ar_cos = map_at_5(pred_cos, gt_val)
print("AR Cos MAP@5:", map_ar_cos)

AR Cos MAP@5: 0.3443862696443341


## Association Rules (Cosine). Weekly baskets

In [118]:
def build_weekly_baskets(airings, threshold=0.8):
    """
    One basket per (user, ISO week).
    Show weight = number of full-watchings in that week.
    """
    a = airings[airings['view_pct'] >= threshold].copy()
    iso = a['show_start'].dt.isocalendar()
    a['week_id'] = iso.year.astype(str) + '_' + iso.week.astype(str)
    grouped = (a.groupby(['user_id', 'week_id', 'tv_show_id'])
               .size().reset_index(name='count'))
    baskets = {}
    for (uid, week), grp in grouped.groupby(['user_id', 'week_id']):
        baskets[(uid, week)] = dict(zip(grp['tv_show_id'], grp['count']))
    return baskets

In [119]:
baskets_users_val = build_baskets(profile_train, min_frequency=1, weight='frequency')
R_users_val, users_val_exp, items_val_exp = build_user_item_matrix(baskets_users_val)

baskets_weekly_val = build_weekly_baskets(airings_train)
R_weekly_val, users_weekly_val, items_weekly_val = build_user_item_matrix(baskets_weekly_val)

S_weekly_val = fit_cosine_rules(R_weekly_val, min_support=5)

pred_weekly_val = recommend_top_k(R_users_val, users_val_exp, items_val_exp,S_weekly_val, val_candidates, k=5)
pred_weekly_val = fill_cold_start(pred_weekly_val, gt_val.keys(), popularity_train)

map_ar_cos_weekly = map_at_5(pred_weekly_val, gt_val)

print("AR Cos MAP@5 weekly baskets:", map_ar_cos_weekly)

AR Cos MAP@5 weekly baskets: 0.3597692307692308


In [120]:
#finetune min support
results = []
for ms in [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 100]:
    S_w = fit_cosine_rules(R_weekly_val, min_support=ms)
    pred = recommend_top_k(R_users_val, users_val_exp, items_val_exp, S_w, val_candidates, k=5)
    pred = fill_cold_start(pred, gt_val.keys(), popularity_train)
    map_ar_cos_weekly = map_at_5(pred, gt_val)
    results.append({'min_support': ms, 'MAP@5': map_ar_cos_weekly})
    
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 min_support    MAP@5
           1 0.359715
           2 0.359811
           3 0.359811
           5 0.359769
           7 0.359825
          10 0.359794
          15 0.359759
          20 0.359711
          30 0.359934
          50 0.360104
         100 0.359595


## Association Rules (Cosine). Weekly baskets + time decay

In [121]:
from scipy.sparse import csr_matrix

def apply_time_decay(R, user_ids, item_ids, airings, half_life_days=30):
    """
    Multiplies the frequencies in matrix R by the damping coefficient
    half_life_days: how many days it takes for the habit weight to drop by half
    """
    t_ref = airings['show_start'].max()
    last_watch = airings.groupby(['user_id', 'tv_show_id'])['show_start'].max().reset_index()
    
    last_watch['decay'] = 0.5 ** ((t_ref - last_watch['show_start']).dt.days / half_life_days)
    
    u_to_idx = {u: i for i, u in enumerate(user_ids)}
    i_to_idx = {s: i for i, s in enumerate(item_ids)}
    
    valid = (last_watch['user_id'].isin(u_to_idx)) & (last_watch['tv_show_id'].isin(i_to_idx))
    vw = last_watch[valid]
    
    rows = [u_to_idx[u] for u in vw['user_id']]
    cols = [i_to_idx[s] for s in vw['tv_show_id']]
    vals = vw['decay'].values
    
    D = csr_matrix((vals, (rows, cols)), shape=R.shape)
    R_decayed = R.multiply(D).tocsr()
    
    return R_decayed

In [122]:
R_users_val_decayed = apply_time_decay(R_users_val, users_val_exp, items_val_exp, airings_train, half_life_days=30)

pred_weekly_decay_val = recommend_top_k(R_users_val_decayed, users_val_exp, items_val_exp,S_weekly_val,val_candidates, k=5)
pred_weekly_decay_val = fill_cold_start(pred_weekly_decay_val, gt_val.keys(), popularity_train)
map_ar_cos_weekly_dacay = map_at_5(pred_weekly_decay_val, gt_val)
print("AR Cos MAP@5 weekly baskets + Time-Decay:", map_ar_cos_weekly_dacay)

AR Cos MAP@5 weekly baskets + Time-Decay: 0.3634044665012407


## Submission for Cosine Asssociation Rules

In [123]:
sessions_trainval = sessions

schedule_trainval = prepare_schedule(pd.concat([arh_11_20, arh_21_30], ignore_index=True))

facts_trainval  = build_facts(sessions_trainval, schedule_trainval)
airings_trainval = build_airings(facts_trainval)
user_show_trainval = build_user_show(airings_trainval)
profile_trainval = build_profile(user_show_trainval)

baskets_trainval = build_baskets(profile_trainval, min_frequency=1, weight='frequency')
R_trainval, users_trainval, items_trainval = build_user_item_matrix(baskets_trainval)
S_cos_trainval = fit_cosine_rules(R_trainval, min_support=5)

test_candidates = arh_31_42.loc[arh_31_42['tv_show_id'] != 0, 'tv_show_id'].unique()
popularity_trainval = build_popularity(user_show_trainval, available_shows=test_candidates, top_n=5)

In [124]:
sub_users = set(submission['user_id'].unique())

pred_test_cos = recommend_top_k(R_trainval, users_trainval, items_trainval, S_cos_trainval, test_candidates, k=5)
pred_test_cos = fill_cold_start(pred_test_cos, sub_users, popularity_trainval)

submission_2 = submission[['user_id']].copy()
submission_2['tv_show_id'] = submission_2['user_id'].map(pred_test_cos)
submission_2['tv_show_id'] = submission_2['tv_show_id'].apply(lambda x: ' '.join(map(str, x)))

submission_2.to_csv('submission_2.csv', index=False)
print(submission_2.head())

               user_id                                    tv_show_id
0  8377619604347126107          2400480 240081 2400508 2400467 20088
1  8381667675275833309         2400480 240081 2400467 200344 2400508
2  8387147770138767246  12001682 12002856 12001734 12001732 12002355
3  8397181578236218580          2400480 240081 20088 2400467 2400508
4  8404698046253197367          2400480 240081 2400467 20088 2400508


## Submission for Cosine Asssociation Rules (weekly baskets)

In [125]:
baskets_weekly_trainval = build_weekly_baskets(airings_trainval)
R_weekly_tv, _, items_weekly_tv = build_user_item_matrix(baskets_weekly_trainval)

S_weekly_final = fit_cosine_rules(R_weekly_tv, min_support=5)

pred_test_weekly = recommend_top_k(R_trainval, users_trainval, items_trainval,S_weekly_final, test_candidates, k=5)
pred_test_weekly = fill_cold_start(pred_test_weekly, sub_users, popularity_trainval)

submission_3 = submission[['user_id']].copy()
submission_3['tv_show_id'] = submission_3['user_id'].map(pred_test_weekly)
submission_3['tv_show_id'] = submission_3['tv_show_id'].apply(lambda x: ' '.join(map(str, x)))

submission_3.to_csv('submission_3.csv', index=False)

## Submission for Cosine Asssociation Rules (weekly baskets + time decay)

In [126]:
R_trainval_decayed = apply_time_decay(R_trainval, users_trainval, items_trainval, airings_trainval, half_life_days=30)

pred_test_decay = recommend_top_k(R_trainval_decayed, users_trainval, items_trainval,S_weekly_final, test_candidates, k=5)
pred_test_decay = fill_cold_start(pred_test_decay, sub_users, popularity_trainval)

submission_4 = submission[['user_id']].copy()
submission_4['tv_show_id'] = submission_4['user_id'].map(pred_test_decay)
submission_4['tv_show_id'] = submission_4['tv_show_id'].apply(lambda x: ' '.join(map(str, x)))
submission_4.to_csv('submission_4.csv', index=False)


## Business & Algorithmic Analysis

In [138]:
results = pd.DataFrame({'Method': ['Popularity Baseline','CBF (Content-Based)','AR (Support)','AR (Confidence)', 'AR (Lift)', 'AR (Cosine, User Baskets)','AR (Cosine, Weekly Baskets)','AR (Cosine, Weekly + Time-Decay)'],
    'MAP@5': [map_popularity,map_cbf_fair,map_ar_support,map_ar_confidence, map_ar_lift,map_ar_cos,map_ar_cos_weekly,map_ar_cos_weekly_dacay]})

display(results)

,Method,MAP@5
0,Popularity Baseline,0.257416
1,CBF (Content-Based),0.147353
2,AR (Support),0.284993
3,AR (Confidence),0.289711
4,AR (Lift),0.016323
5,"AR (Cosine, User Baskets)",0.344386
6,"AR (Cosine, Weekly Baskets)",0.359595
7,"AR (Cosine, Weekly + Time-Decay)",0.363404


In [139]:
def get_user_ap(pred, gt, uid):
    """Calculates AP@5 for one user"""
    hits = 0
    ap = 0.0
    for k, sid in enumerate(pred.get(uid, [])[:5], start=1):
        if sid in gt[uid]:
            hits += 1
            ap += hits / k
    return ap / 5.0

In [143]:
#let us get 2 users 
u1 = 11614703798567549663
u2 = 3223027569404031218

In [164]:
title_col = 'tv_show_name' if 'tv_show_name' in val_meta.columns else 'tv_show_title'
meta = val_meta[[title_col, 'tv_show_category', 'duration']].to_dict('index')

def print_simple_analysis(user_id, methods_dict):
    rows = []
    hist_set = set(profile_train.get(user_id, {}).keys())
    
    for method_name, preds in methods_dict.items():
        top5 = preds.get(user_id, [])
        for rank, sid in enumerate(top5, 1):
            m = meta.get(sid, {})
            dur_sec = m.get('duration', 0)
            dur_mins = round(dur_sec / 60, 1) if pd.notna(dur_sec) else 0.0
            
            rows.append({
                'Method': method_name,
                'Rank': rank,
                'ID': sid,
                'Title': str(m.get(title_col, 'Unknown'))[:30],
                'Category': m.get('tv_show_category', '-'),
                'Dur (mins)': dur_mins,
                'Popularity': pop_dict.get(sid, 0), 
                'In History?': 'Yes' if sid in hist_set else 'No'
            })
            
    print(f"\n=== USER ID: {user_id} ===")
    display(pd.DataFrame(rows))

# Обновленный словарь методов (синхронизирован с таблицей AP@5)
methods = {
    'Popularity Baseline': pred_val_popularity,
    'CBF (Content-Based)': pred_val_cbf,
    'AR (Confidence - Classic)': pred_conf,
    'AR-based CF (Cosine + Decay)': pred_weekly_decay_val
}

print_simple_analysis(u1, methods)
print_simple_analysis(u2, methods)

,Method,User 1 AP@5,User 2 AP@5,User 1 Top-5,User 2 Top-5,User 1 True ans,User 2 True ans
0,Popularity Baseline,0.180,0.55,"[2400480, 240081, 2400467, 200344, 2400508]","[2400480, 240081, 2400467, 200344, 2400508]","[2400508, 5900256, 200432, 200469, 240081]","[2400480, 1000589, 20088, 200344, 240081]"
1,CBF (Content-Based),0.520,0.00,"[2400508, 5900256, 2400480, 5900347, 240081]","[90079432, 90081815, 90063059, 90049108, 90024441]","[2400508, 5900256, 200432, 200469, 240081]","[2400480, 1000589, 20088, 200344, 240081]"
2,AR (Confidence),0.233,0.55,"[2400480, 240081, 2400508, 2400467, 200344]","[2400480, 240081, 2400467, 200344, 2400508]","[2400508, 5900256, 200432, 200469, 240081]","[2400480, 1000589, 20088, 200344, 240081]"
3,AR-based CF (Cosine + Decay),0.200,1.00,"[2400480, 2400508, 20088, 240081, 2400467]","[2400480, 240081, 1000589, 200344, 20088]","[2400508, 5900256, 200432, 200469, 240081]","[2400480, 1000589, 20088, 200344, 240081]"


In [166]:
title_col = 'tv_show_name' if 'tv_show_name' in val_meta.columns else 'tv_show_title'
meta = val_meta[[title_col, 'tv_show_category', 'duration']].to_dict('index')

def print_simple_analysis(user_id, methods_dict):
    rows = []
    hist_set = set(profile_train.get(user_id, {}).keys())
    
    for method_name, preds in methods_dict.items():
        top5 = preds.get(user_id, [])
        for rank, sid in enumerate(top5, 1):
            m = meta.get(sid, {})
            dur_sec = m.get('duration', 0)
            dur_mins = round(dur_sec / 60, 1) if pd.notna(dur_sec) else 0.0
            
            rows.append({
                'Method': method_name,
                'Rank': rank,
                'ID': sid,
                'Title': str(m.get(title_col, 'Unknown'))[:30],
                'Category': m.get('tv_show_category', '-'),
                'Dur (mins)': dur_mins,
                'Popularity': pop_dict.get(sid, 0), 
                'In History?': 'Yes' if sid in hist_set else 'No'
            })
            
    print(f"\n=== USER ID: {user_id} ===")
    display(pd.DataFrame(rows))

methods = {
    'Popularity Baseline': pred_val_popularity,
    'CBF (Content-Based)': pred_val_cbf,
    'AR (Confidence)': pred_conf,
    'AR-based CF (Cosine + Decay)': pred_weekly_decay_val
}

print_simple_analysis(u1, methods)
print_simple_analysis(u2, methods)


=== USER ID: 11614703798567549663 ===


,Method,Rank,ID,Title,Category,Dur (mins),Popularity,In History?
0,Popularity Baseline,1,2400480,ТСН.,Инфо,10.0,4492,Yes
1,Popularity Baseline,2,240081,Факты. Утро.,Инфо,25.0,4269,Yes
2,Popularity Baseline,3,2400467,'Викна-Новости'.,Инфо,20.0,3842,Yes
3,Popularity Baseline,4,200344,'ЧистоNews'.,Развлечения,10.0,3025,No
4,Popularity Baseline,5,2400508,Сегодня.,Инфо,30.0,3978,Yes
5,CBF (Content-Based),1,2400508,Сегодня.,Инфо,30.0,3978,Yes
6,CBF (Content-Based),2,5900256,Утро с Украиной.,Инфо,30.0,2067,Yes
7,CBF (Content-Based),3,2400480,ТСН.,Инфо,10.0,4492,Yes
8,CBF (Content-Based),4,5900347,'Деньги'.,Инфо,80.0,1885,No
9,CBF (Content-Based),5,240081,Факты. Утро.,Инфо,25.0,4269,Yes



=== USER ID: 3223027569404031218 ===


,Method,Rank,ID,Title,Category,Dur (mins),Popularity,In History?
0,Popularity Baseline,1,2400480,ТСН.,Инфо,10.0,4492,Yes
1,Popularity Baseline,2,240081,Факты. Утро.,Инфо,25.0,4269,Yes
2,Popularity Baseline,3,2400467,'Викна-Новости'.,Инфо,20.0,3842,No
3,Popularity Baseline,4,200344,'ЧистоNews'.,Развлечения,10.0,3025,Yes
4,Popularity Baseline,5,2400508,Сегодня.,Инфо,30.0,3978,No
5,CBF (Content-Based),1,90079432,Х/ф 'Портниха: Месть от кутюр',Фильмы,130.0,490,No
6,CBF (Content-Based),2,90081815,Х/ф 'Букшоп'.,Фильмы,115.0,349,No
7,CBF (Content-Based),3,90063059,Х/ф 'Иллюзионист'.,Фильмы,115.0,577,No
8,CBF (Content-Based),4,90049108,Х/ф 'Поездка в Индию'.,Фильмы,165.0,62,No
9,CBF (Content-Based),5,90024441,Х/ф 'Филадельфия'.,Фильмы,165.0,93,No


In [161]:
cat_cols = [c for c in user_vec_train.columns if c.startswith('cat_')]

vec_u1 = user_vec_train.loc[u1, cat_cols].rename('User 1')
vec_u2 = user_vec_train.loc[u2, cat_cols].rename('User 2')

df_vectors = pd.concat([vec_u1, vec_u2], axis=1).sort_values(by='User 1', ascending=False)

df_vectors = df_vectors[(df_vectors > 0).any(axis=1)]

display(df_vectors.round(3))

,User 1,User 2
cat_Инфо,0.554,0.25
cat_Развлечения,0.297,0.15
cat_Детям,0.068,0.05
cat_Сериалы,0.054,0.25
cat_Фильмы,0.027,0.30
